In [ ]:
!pip install -q \
langchain==0.1.16 \
langchain-community==0.0.36 \
sentence-transformers \
transformers \
faiss-cpu \
pypdf \
accelerate

In [ ]:
import os #used to iterate files over a directory
from langchain_community.document_loaders import PyPDFLoader #to extract text from pdf files
from langchain.text_splitter import RecursiveCharacterTextSplitter #split large documents into small chunks
from langchain_community.embeddings import HuggingFaceEmbeddings #text to dense vector embeddings
from langchain_community.vectorstores import FAISS #stores and searches for vectors

DATA_PATH = "data/documents"

documents = []
for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load()) #extracted text is stored as "documents" with metadata

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) #500 character chunks, 50 character-overlap to ensure semantic continuity
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
) #converts each chunk into a high-dimensional vector

db = FAISS.from_documents(chunks, embeddings) #FAISS indexes the embeddings for fast similarity search (FAISS acts as retrieval backend)
db.save_local("faiss_index") #indexes are saved locally to avoid reuse during querying

print(f"Loaded {len(documents)} pages")
print(f"Created {len(chunks)} chunks")

In [ ]:
#construction of RAG pipeline - connects vector database and LLM
from langchain.chains import RetrievalQA #combines retrieval and generation
from langchain_community.vectorstores import FAISS #loads vector index
from langchain_community.embeddings import HuggingFaceEmbeddings #to embed user queries
from langchain_community.llms import HuggingFacePipeline #wraps hugging face models into a langchain-compatible LLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline #loads and runs the open-source LLM

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)#same embedding model as ingestion step

db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
) #loads pre-computed FAISS index from disk

model_name = "google/flan-t5-base" #instruction-based model, works well on QnA tasks

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512
)#text generation pipeline

llm = HuggingFacePipeline(pipeline=pipe) #bridge between hugging face and langchain

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
) #user query is embedded -> top 3 relevant chunks are retrieved -> retrieved text is passed to LLM as context -> LLM generates a grounded answer -> source documents are returned

print("RAG system ready")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output #both tools enable event-driven interaction

#input box
query_input = widgets.Text(
    placeholder="Enter your question here",
    description="Question:",
    layout=widgets.Layout(width="80%")
)

# ask button
ask_button = widgets.Button(
    description="Ask",
    button_style="primary"
)

# o/p area
output = widgets.Output()

# Define button click behavior
def on_ask_clicked(b):
    with output:
        clear_output()
        query = query_input.value.strip() #clear previous output

        if not query:
            print("Please enter a question.")
            return

        print("Processing...\n")
        result = qa(query)

        print("Answer:\n")
        print(result["result"])
        print("\nSources:")
        for doc in result["source_documents"]:
            print("-", doc.metadata.get("source", "Unknown document"))

ask_button.on_click(on_ask_clicked) #links button click event to function

display(query_input, ask_button, output) #displays UI